In [2]:
df = spark.read.parquet("abfss://btc-transactions@btcetlstorage.dfs.core.windows.net/")

print(f"Total rows: {df.count()}")
print(f"Columns: {df.columns}")
df.show(5)

StatementMeta(etlsparkpool, 2, 3, Finished, Available, Finished, False)

Total rows: 8781456
Columns: ['txid', 'hash', 'version', 'size', 'block_hash', 'block_number', 'index', 'virtual_size', 'lock_time', 'input_count', 'output_count', 'is_coinbase', 'output_value', 'outputs', 'block_timestamp', 'date', 'last_modified', 'fee', 'input_value', 'inputs']
+--------------------+--------------------+-------+----+--------------------+------------+-----+------------+---------+-----------+------------+-----------+-------------------+--------------------+-------------------+----------+--------------------+------+-------------------+--------------------+
|                txid|                hash|version|size|          block_hash|block_number|index|virtual_size|lock_time|input_count|output_count|is_coinbase|       output_value|             outputs|    block_timestamp|      date|       last_modified|   fee|        input_value|              inputs|
+--------------------+--------------------+-------+----+--------------------+------------+-----+------------+---------+---

big picture

df (8.7M rows, 20 columns, raw)
        ↓
.select() → 9 important columns
        ↓
.dropna() → remove missing data
        ↓
.filter() → remove mining transactions
        ↓
.withColumn() × 3 → add fee%, complexity, category
        ↓
df_clean → ready for analysis! ✅

In [3]:
from pyspark.sql.functions import col, round, when

# Select important columns
df_clean = df.select(
    "txid",
    "block_timestamp",
    "block_number",
    "output_value",
    "input_value",
    "fee",
    "input_count",
    "output_count",
    "is_coinbase"
)

StatementMeta(etlsparkpool, 2, 4, Finished, Available, Finished, False)

removes mining reward transactions. we only want real person-to-person transfers for our analytics.

In [4]:

# Drop nulls
df_clean = df_clean.dropna(subset=["txid", "output_value", "block_timestamp"])

# Filter out coinbase transactions
df_clean = df_clean.filter(col("is_coinbase") == False)

StatementMeta(etlsparkpool, 2, 5, Finished, Available, Finished, False)

what is fee percentage?
every Bitcoin transaction pays a small fee to miners who process it. fee percentage tells us:

"what % of the total amount sent was paid as fee?"

example:

input_value = 1.0 BTC (you're sending this)
fee = 0.0001 BTC (you pay this to miners)
fee_percentage = 0.01%

Complexity ratio
tells us how many receivers per sender:

ratio = 1.0 → simple, 1 person sending to 1 person
ratio = 10.0 → complex, 1 person sending to 10 people (like a company paying employees)

In [5]:
# New column 1: fee percentage
df_clean = df_clean.withColumn(
    "fee_percentage",
    round((col("fee") / col("input_value")) * 100, 4)
)

# New column 2: complexity ratio
df_clean = df_clean.withColumn(
    "complexity_ratio",
    round(col("output_count") / col("input_count"), 4)
)

# New column 3: transaction size category
df_clean = df_clean.withColumn(
    "btc_value_category",
    when(col("output_value") > 50, "whale")
    .when(col("output_value") > 10, "large")
    .when(col("output_value") > 1, "medium")
    .otherwise("small")
)

StatementMeta(etlsparkpool, 2, 6, Finished, Available, Finished, False)

In [6]:
print(f"Rows after cleaning: {df_clean.count()}")
df_clean.show(5)

StatementMeta(etlsparkpool, 2, 7, Finished, Available, Finished, False)

Rows after cleaning: 8772036
+--------------------+-------------------+------------+-------------------+-------------------+------+-----------+------------+-----------+--------------+----------------+------------------+
|                txid|    block_timestamp|block_number|       output_value|        input_value|   fee|input_count|output_count|is_coinbase|fee_percentage|complexity_ratio|btc_value_category|
+--------------------+-------------------+------------+-------------------+-------------------+------+-----------+------------+-----------+--------------+----------------+------------------+
|7cdd3ad4ab269d2f4...|2014-07-01 03:24:17|      308684|         0.04980851|         0.04990851|1.0E-4|          1|           2|      false|        0.2004|             2.0|             small|
|b6b7f17318b70be8f...|2014-07-01 20:10:27|      308780|            0.11257|            0.11267|1.0E-4|          1|           2|      false|        0.0888|             2.0|             small|
|3f97ec09c4c62a9

In [7]:
df_clean.groupBy("btc_value_category").count().show()

StatementMeta(etlsparkpool, 2, 8, Finished, Available, Finished, False)

+------------------+-------+
|btc_value_category|  count|
+------------------+-------+
|             whale| 307484|
|            medium|1757882|
|             small|6100875|
|             large| 605795|
+------------------+-------+



In [8]:
df_clean.filter(col("btc_value_category")=="whale").show(10)

StatementMeta(etlsparkpool, 2, 9, Finished, Available, Finished, False)

+--------------------+-------------------+------------+------------------+------------------+------+-----------+------------+-----------+--------------+----------------+------------------+
|                txid|    block_timestamp|block_number|      output_value|       input_value|   fee|input_count|output_count|is_coinbase|fee_percentage|complexity_ratio|btc_value_category|
+--------------------+-------------------+------------+------------------+------------------+------+-----------+------------+-----------+--------------+----------------+------------------+
|92b6a780944477acd...|2014-07-01 01:43:56|      308679|      100.03114487|100.03174487000001|6.0E-4|         25|          19|      false|        6.0E-4|            0.76|             whale|
|646a5a1228a62b183...|2014-07-01 21:31:50|      308789|       56.46025242|56.460752420000006|5.0E-4|          1|           3|      false|        9.0E-4|             3.0|             whale|
|59ae4d68739bf80f4...|2014-07-01 18:14:51|      308773|

In [9]:
# Count of each category
df_clean.groupBy("btc_value_category").count().orderBy("count", ascending=False).show()

StatementMeta(etlsparkpool, 2, 10, Finished, Available, Finished, False)

+------------------+-------+
|btc_value_category|  count|
+------------------+-------+
|             small|6100875|
|            medium|1757882|
|             large| 605795|
|             whale| 307484|
+------------------+-------+



In [10]:
# Save cleaned data back to Azure Data Lake
df_clean.write.mode("overwrite") \
    .parquet("abfss://btc-transactions@btcetlstorage.dfs.core.windows.net/cleaned/")

print("✅ Cleaned data saved to Azure Data Lake!")

StatementMeta(etlsparkpool, 2, 11, Finished, Available, Finished, False)

✅ Cleaned data saved to Azure Data Lake!


Cleaned Data (Azure)
        ↓
Kafka Producer → reads cleaned parquet
              → sends each transaction as a message
        ↓
Kafka Topic (like a queue) → "btc-transactions"
        ↓
Kafka Consumer → receives messages
              → simulates real-time processing

In [11]:
# Take a sample of 500k rows for Power BI (manageable size)
df_sample = df_clean.sample(fraction=0.06, seed=42)
print(f"Sample size: {df_sample.count()}")

StatementMeta(etlsparkpool, 2, 12, Finished, Available, Finished, False)

Sample size: 526439


In [12]:
import pandas as pd
from azure.storage.blob import BlobServiceClient

# Convert to pandas and save as CSV
print("Converting to pandas...")
df_pandas = df_sample.toPandas()

print("Saving CSV...")
df_pandas.to_csv("/tmp/btc_clean.csv", index=False)

# Upload to Azure
print("Uploading to Azure...")
conn_str = "DefaultEndpointsProtocol=https;AccountName=btcetlstorage;AccountKey=YOUR_AZURE_CONNECTION_STRING_HERE;EndpointSuffix=core.windows.net"

blob_service = BlobServiceClient.from_connection_string(conn_str)
blob_client = blob_service.get_blob_client(
    container="btc-transactions", 
    blob="btc_clean.csv"
)

with open("/tmp/btc_clean.csv", "rb") as f:
    blob_client.upload_blob(f, overwrite=True)

print("✅ CSV uploaded to Azure! Ready for Power BI!")

StatementMeta(etlsparkpool, 2, 13, Finished, Available, Finished, False)

Converting to pandas...
Saving CSV...
Uploading to Azure...
✅ CSV uploaded to Azure! Ready for Power BI!
